# SupportAI — NLP : exécution complète sur Google Colab
Ce notebook **clone le dépôt, entraîne les 3 modèles avec de vrais embeddings (SBERT multilingue), lance les tests et prépare un zip de résultats** à remettre dans le dépôt.

Pourquoi Colab : l'entraînement SBERT nécessite un accès à Hugging Face, et certains PC Windows bloquent les bibliothèques Python (pandas/scikit-learn).

**Utilisation : menu Exécution → Tout exécuter** (≈ 5 à 10 minutes). Aucun GPU nécessaire.

In [ ]:
import os
REPO_URL = os.environ.get("SUPPORTAI_REPO", "https://github.com/Houcineaberhache/Projet-Data-Science-SupportAI")
BRANCH   = os.environ.get("SUPPORTAI_BRANCH", "main")
if not os.path.isdir("repo"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} repo
%cd repo/ml/nlp
!git log --oneline -1

In [ ]:
# Ne pas toucher à numpy / scikit-learn de Colab : on n'installe que le manquant.
!pip install -q fastapi uvicorn httpx pytest sentence-transformers
import sklearn, pandas, sentence_transformers
print("scikit-learn", sklearn.__version__, "| pandas", pandas.__version__, "| sentence-transformers", sentence_transformers.__version__)

## 1. Vérifier que nos intentions = les 14 intentions officielles

In [ ]:
!python -m intent_clf.check_intents

## 2. Entraîner et évaluer (TF-IDF+LR, Linear SVM, embeddings)

In [ ]:
!python -m intent_clf.train 2>&1 | grep -v -i warn

In [ ]:
import json
cfg = json.load(open("models/config.json"))
print(cfg["embedding_backend"], "->", cfg["best_model"])
if cfg["embedding_backend"] != "sbert":
    print("\n⚠️ ATTENTION : Hugging Face inaccessible, repli LSA utilisé. Les résultats 'embeddings' ne sont PAS de vrais embeddings.")
else:
    print("\n✅ Vrais embeddings (paraphrase-multilingual-MiniLM-L12-v2) utilisés.")

## 3. Tests de l'API

In [ ]:
!python -m pytest -q

## 4. Tableau de résultats (à coller dans le README)

In [ ]:
m = json.load(open("reports/metrics.json"))
names = {"tfidf_lr": "TF-IDF + LogReg", "tfidf_svm": "TF-IDF + Linear SVM",
         "embed_lr": "Embeddings (SBERT) + LogReg" if cfg["embedding_backend"] == "sbert" else "Embeddings + LogReg (repli LSA)"}
best = cfg["best_model"]
rows = ["| Modèle | CV groupée F1 macro | Test manuel (140) accuracy | Test manuel F1 macro | Pilot 20 accuracy |", "|---|---|---|---|---|"]
f = lambda x, d=3: f"{x:.{d}f}".replace(".", ",")
for k, v in m.items():
    n = f"**{names[k]}**" if k == best else names[k]
    rows.append(f"| {n} | {f(v['cv_grouped']['f1_macro'])} | {f(v['test_manual']['accuracy'])} | {f(v['test_manual']['f1_macro'])} | {f(v['pilot_20']['accuracy'], 2)} |")
print("\n".join(rows))

## 5. Régénérer le notebook comparatif avec ces résultats

In [ ]:
!jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=900 notebooks/model_comparison.ipynb 2>&1 | tail -2

## 6. Zip des résultats à remettre dans le dépôt

In [ ]:
import zipfile
files = ["models/config.json", "notebooks/model_comparison.ipynb"] + [f"reports/{f}" for f in os.listdir("reports")]
ZIP = "/content/nlp_results_sbert.zip" if os.path.isdir("/content") else os.path.abspath("nlp_results_sbert.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=f"ml/nlp/{f}")   # chemins relatifs à la RACINE du dépôt
print("Zip créé :", ZIP, "-", len(files), "fichiers")
try:
    from google.colab import files as colab_files
    colab_files.download(ZIP)
except Exception as e:
    print("(téléchargement automatique indisponible hors Colab)", type(e).__name__)

## Et ensuite (sur votre PC)
1. Décompressez `nlp_results_sbert.zip` **à la racine du dépôt** (il écrase `ml/nlp/reports`, `ml/nlp/models/config.json` et le notebook comparatif).
2. Collez le tableau de la section 4 dans `ml/nlp/README.md` (section Résultats) et retirez la mention « repli LSA » des Limites si le backend était `sbert`.
3. `git add ml/nlp` → `git commit -m "docs(nlp): results with real SBERT embeddings"` → `git push` : la pull request se met à jour toute seule.